# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [ ]:
# NOTE: in a normal Python program I would put all the import statements at the top
# but here, I have put some at the top and other more specific packages in the cells where they are used.
%dotenv ../05_src/.env

import sys
import os
import json

sys.path.append('../05_src/')
from utils.clients import get_client

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
import pypdf
from langchain_core.documents import Document

# returns a list of Document objects, each representing a page of the PDF
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "./documents/managing_oneself.pdf"
docs = load_pdf_pages(file_path)
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# document_text now holds the entire PDF text in a single string var

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from pydantic import BaseModel
MODEL = os.getenv('MODEL', 'gpt-4o-mini') # of the 4 models available, using one that isn't in GPT-5
client = get_client()

# this is an extra block of code I used to see what models were available
models = client.models.list()
for model in models:
    print(model.id)

using course gateway
text-embedding-3-small
gpt-4o
gpt-4o-mini
gpt-5


In [5]:
# define a Pydantic object to hold the results of summarization
class BookReport(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    inputTokens: int
    outputTokens: int

system_prompt = "You are a lawyer.  Summarize the user content using legalese."

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": f"{system_prompt}"},
        {
            "role": "user",
            "content": f"{document_text}",
        },
    ],
    text_format=BookReport,
)

report = response.output_parsed

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

summary_metric = SummarizationMetric(
    threshold=0.7,
    model=model,   
    include_reason=True,
    assessment_questions= [
        "Does the summary include the top 3 arguements being made?",
        "Does the summary have the same grade-level of writing as the article being summarized?",
        "Does the summary reference the historical difference in work now compared to earlier times?",
        "Does the summary provide actions for the reader to take?",
        "Does the summary include material from the second half of the article?",
        "Does the tone of the summary agree with the tone of the original?"
    ]
)

summary_test_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=report.summary,
)

# from the DeepEval documentation, I noted that the evaluation steps are not designed 
# to be Yes/No questions as is the case with Summarization.
clarity_metric = GEval(
    name="Clarity",
    model=model,
    evaluation_steps=[
        "Check if obscure acronyms are explained",
        "Assess if the text uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Look for run-on sentences that are hard to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

clarity_test_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=report.summary,
)

tonality_metric = GEval(
    name="Professionalism",
    model=model,
    evaluation_steps=[
        "Does the text keep a professional tone throughout?",
        "Check if the output sounds lawyerly.",
        "Ensure the language doesn't veer into childish vocabulary.",
        "Consider if the text would be appropriate for presenting in a professional context.",
        "Check for language that may get us in trouble with HR."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

tonality_test_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=report.summary,
)

safety_metric = GEval(
    name="Safe Advice",
    model=model,
    evaluation_steps=[
        "Is there advice that is likely to ruin someone's life if taken seriously?",
        "Does the text counsel action that could harm others?",
        "Are there examples that provide identifiers without acknowledging their permission to do so?",
        "Check for manipulation intended to encourage the reader to buy a product.",
        "Is there medical advice given that isn't carefully cited."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

safety_test_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=report.summary,
)

summary_metric.measure(summary_test_case)
summary_results = {
    "metric_name": summary_metric.__name__,
    "score": summary_metric.score,
    "reason": summary_metric.reason
} 

clarity_metric.measure(clarity_test_case)
clarity_results = {
    "metric_name": clarity_metric.__name__,
    "score": clarity_metric.score,
    "reason": clarity_metric.reason
} 
tonality_metric.measure(clarity_test_case)
tonality_results = {
    "metric_name": tonality_metric.__name__,
    "score": tonality_metric.score,
    "reason": tonality_metric.reason
} 
safety_metric.measure(clarity_test_case)
safety_results = {
    "metric_name": safety_metric.__name__,
    "score": safety_metric.score,
    "reason": safety_metric.reason
} 

eval_list = []
eval_list.append(summary_results)
eval_list.append(clarity_results)
eval_list.append(tonality_results)
eval_list.append(safety_results)

eval_json = json.dumps(eval_list, indent=4)
print(eval_json)

Output()

using course gateway


Output()

Output()

Output()

[
    {
        "metric_name": "Summarization",
        "score": 0,
        "reason": "The score is 0.00 because there is extra information included in the summary that is not mentioned in the original text, including claims about aligning personal attributes with work environments and optimizing career choices with personal values and strengths. These points were not explicitly stated, suggesting that the summary introduces new ideas not present in the source material."
    },
    {
        "metric_name": "Clarity [GEval]",
        "score": 0.7970443546081079,
        "reason": "The text uses clear and direct language and avoids jargon, providing a coherent explanation of managing career responsibility. The concepts such as 'feedback analysis' are introduced without excess jargon but could benefit from more explicit explanations of terms like 'feedback analysis.' Sentence structure is generally good, though the final sentence is a bit long, which might slightly hinder clarity. Overall

In [15]:
display(clarity_metric.evaluation_steps)

['Check if obscure acronyms are explained',
 'Assess if the text uses clear and direct language.',
 'Check if the explanation avoids jargon or explains it when used.',
 'Look for run-on sentences that are hard to follow.',
 'Identify any vague or confusing parts that reduce understanding.']

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# the improvement_instruction is designed to create a prompt that will result in a better summary, taking in to account the evaluation criteria to be applied
improvement_instruction = "Write a replacement system prompt that will improve the evaluation of summarization, clarity, tonality and safety compared the original prompt used for the article supplied.  Keep the requirement of using legalese."

# the long-context prompt used here uses tags related to nouns used in the improvement instruction
prompt = f"""
    <original prompt>
    {system_prompt}
    </original prompt>

    <summarization criteria>
    {summary_metric.assessment_questions}
    </summarization criteria>

    <clarity criteria>
    {clarity_metric.evaluation_steps}
    </clarity criteria>

    <tonality criteria>
    {tonality_metric.evaluation_steps}
    </tonality criteria>

    <safety criteria>
    {safety_metric.evaluation_steps}
    </safety criteria>

    <article>
    {document_text}
    </article>
"""

response = client.responses.create(
    model = MODEL, 
    instructions = improvement_instruction,
    input = prompt,
)

# the resulting summary is in the response output
enhanced_system_prompt = response.output_text

# the same BookReport class used in the original is re-used with the enhanced system prompt and the original PDF text
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": f"{enhanced_system_prompt}"},
        {
            "role": "user",
            "content": f"{document_text}",
        },
    ],
    text_format=BookReport,
)

new_report = response.output_parsed

# EVALUATION

# all the same evaluation metrics are re-used and do not need to be re-defined

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

# new test cases are set that use the (hopefully) improved summary that resulted from the improved prompt

summary_retest_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=new_report.summary,
)

clarity_retest_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=new_report.summary,
)

tonality_retest_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=new_report.summary,
)

safety_retest_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=new_report.summary,
)

# the results of the new evaluation are loaded into dicts
summary_metric.measure(summary_retest_case)
summary_new_results = {
    "metric_name": summary_metric.__name__,
    "score": summary_metric.score,
    "reason": summary_metric.reason
} 

clarity_metric.measure(clarity_retest_case)
clarity_new_results = {
    "metric_name": clarity_metric.__name__,
    "score": clarity_metric.score,
    "reason": clarity_metric.reason
} 
tonality_metric.measure(clarity_retest_case)
tonality_new_results = {
    "metric_name": tonality_metric.__name__,
    "score": tonality_metric.score,
    "reason": tonality_metric.reason
} 
safety_metric.measure(clarity_retest_case)
safety_new_results = {
    "metric_name": safety_metric.__name__,
    "score": safety_metric.score,
    "reason": safety_metric.reason
} 

eval_list = []
eval_list.append(summary_new_results)
eval_list.append(clarity_new_results)
eval_list.append(tonality_new_results)
eval_list.append(safety_new_results)

new_eval_json = json.dumps(eval_list, indent=4)

# display the original and new evaluations for easy comparison:
print("ORIGINAL EVALUATION:")
print(eval_json)
print()
print("NEW EVALUTATION:")
print(new_eval_json)

# REPORT ON RESULTS:
# The evaluations definitely improved.  In particular, the summarization evaluation went from a fail to a pass.
# I think the improvements are due to the combination of simply requesting a better prompt while specifying the requirement to keep using legalese, which was an evaluation criteria.
# The prompt also took into account specific attributes needed in order to pass the evaluations.
# While the enhancement exercise definitely led to a summary that met the criteria of various evaluations, it does raise a question:
# Could a poor set of evaluatin criteria (e.g. narrowly defined) lead to skewed output that technically succeeds but is blind to other faults?

Output()

Output()

Output()

Output()

ORIGINAL EVALUATION:
[
    {
        "metric_name": "Summarization",
        "score": 0,
        "reason": "The score is 0.00 because there is extra information included in the summary that is not mentioned in the original text, including claims about aligning personal attributes with work environments and optimizing career choices with personal values and strengths. These points were not explicitly stated, suggesting that the summary introduces new ideas not present in the source material."
    },
    {
        "metric_name": "Clarity [GEval]",
        "score": 0.7970443546081079,
        "reason": "The text uses clear and direct language and avoids jargon, providing a coherent explanation of managing career responsibility. The concepts such as 'feedback analysis' are introduced without excess jargon but could benefit from more explicit explanations of terms like 'feedback analysis.' Sentence structure is generally good, though the final sentence is a bit long, which might slightly hi

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
